In [1]:
import matplotlib.pyplot as plt
import numpy as np
from backend import _solve_kepler

In [2]:
# Sky projected orbit plotter function
def get_sky_projected_orbit(a, e, i, omega, Omega, num_points):

    # Computes sky-projected coordinates (X, Y) from Keplerian elements. Angles (i, omega, Omega) should be in degrees.
    
    # Convert angles to radians
    i_rad = np.radians(i)
    omega_rad = np.radians(omega)
    Omega_rad = np.radians(Omega)
    
    # 1. Sample true anomalies across a full orbit
    nu = np.linspace(0, 2 * np.pi, num_points)
    
    # 2. Distance 'r' from the focus for an ellipse
    r = (a * (1 - e**2)) / (1 + e * np.cos(nu))
    
    # Position vectors in the intrinsic orbital plane
    x_prime = r * np.cos(nu)
    y_prime = r * np.sin(nu)
    
    # 3. Apply Thiele-Innes or standard rotation matrix combinations
    # X (East) and Y (North) positions projected on the plane of the sky
    X = x_prime * (np.cos(omega_rad) * np.sin(Omega_rad) + np.sin(omega_rad) * np.cos(Omega_rad) * np.cos(i_rad)) + \
        y_prime * (-np.sin(omega_rad) * np.sin(Omega_rad) + np.cos(omega_rad) * np.cos(Omega_rad) * np.cos(i_rad))
        
    Y = x_prime * (np.cos(omega_rad) * np.cos(Omega_rad) - np.sin(omega_rad) * np.sin(Omega_rad) * np.cos(i_rad)) + \
        y_prime * (-np.sin(omega_rad) * np.cos(Omega_rad) - np.cos(omega_rad) * np.sin(Omega_rad) * np.cos(i_rad))
        
    return X, Y



def sky_position_at_epoch(t, P, T, e, a, i, omega, Omega):
    """
    Model position on the sky at epoch(s) t, using the same projection as
    get_sky_projected_orbit(). 'a' in the units you want back (mas here);
    angles in degrees; t, T, P all in years.
    Returns (North, East) so it plots directly on your (x = North, y = East) axes.
    """
    t = np.atleast_1d(np.asarray(t, dtype=float))
 
    # Mean anomaly at each epoch
    M = 2 * np.pi * (((t - T) / P) % 1.0)
 
    # Eccentric anomaly from PySVOrbit's own Kepler solver (one epoch at a time)
    E = np.array([_solve_kepler(m, e) for m in M])
 
    # Safety check: _solve_kepler can occasionally diverge at very high e
    bad = np.abs(E - e * np.sin(E) - M) > 1e-6
    if bad.any():
        raise RuntimeError(f"_solve_kepler did not converge for epoch(s) {t[bad]}")
 
    # True anomaly and orbital radius
    nu = 2 * np.arctan2(np.sqrt(1 + e) * np.sin(E / 2), np.sqrt(1 - e) * np.cos(E / 2))
    r = a * (1 - e**2) / (1 + e * np.cos(nu))
    xp, yp = r * np.cos(nu), r * np.sin(nu)
 
    io, wo, Oo = np.radians([i, omega, Omega])
    East = xp * (np.cos(wo) * np.sin(Oo) + np.sin(wo) * np.cos(Oo) * np.cos(io)) + \
           yp * (-np.sin(wo) * np.sin(Oo) + np.cos(wo) * np.cos(Oo) * np.cos(io))
    North = xp * (np.cos(wo) * np.cos(Oo) - np.sin(wo) * np.sin(Oo) * np.cos(io)) + \
            yp * (-np.sin(wo) * np.cos(Oo) - np.cos(wo) * np.sin(Oo) * np.cos(io))
    return North, East


In [3]:
# Taking any chi2r under 4
data1 = np.genfromtxt("PMOIRED_chi2r/2014_07_29_chi2r_sep_PA.csv", delimiter=",", skip_header=1, usecols = (0, 1, 2))

chi2r_1 = data1[:, 0]  
sep_1 = data1[:, 1]                
PA_deg_1 = data1[:, 2]   

PA_1 = np.deg2rad(PA_deg_1)

x_obs_1 = sep_1 * np.cos(PA_1)
y_obs_1 = sep_1 * np.sin(PA_1)

In [4]:
# --------------------------------------_
# Check for the PMOIRED 2014-7-29 Fits that are on the Monte-Carlo Orbits and Tokovinin 2016
# Keep 2014-07-29 rows with separation in [7, 9] mas and position angle in [260, 280] deg.
keep = (sep_1 >= 7) & (sep_1 <= 9) & (PA_deg_1 >= 265) & (PA_deg_1 <= 280)
chi2r_filt = chi2r_1[keep].tolist()
sep_filt = sep_1[keep].tolist()
PA_filt = PA_deg_1[keep].tolist()



In [5]:
print(np.median(sep_filt))
print(np.median(PA_filt))

print(f'2014-07-29 Separation\n  {np.median(sep_filt)} ± {np.std(sep_filt, ddof=1)/np.sqrt(len(sep_filt))} mas')
print(f'2014-07-29 Position Angle\n  {np.median(PA_filt)} deg')


7.858957634192482
276.94598854237313
2014-07-29 Separation
  7.858957634192482 ± 0.04908736257956782 mas
2014-07-29 Position Angle
  276.94598854237313 deg


In [6]:
# BREAKER